# 10. ComfyUI → code tracing → измеримый архитектурный эксперимент

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 1](https://diffusion.csail.mit.edu/2026/docs/20260120_Lecture_01.pdf) · [Lecture 2](https://diffusion.csail.mit.edu/2026/docs/20260122_Lecture_02.pdf) · [Lecture 3](https://diffusion.csail.mit.edu/2026/docs/20260123_Lecture_03.pdf) · [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Свяжем визуальный graph с разобранными hidden blocks, составим ручной measurement protocol и отделим функциональные CPU проверки от real inference.

[Маршрут](../README.md) · [Точные задания MIT](../MIT_BRIDGE.md) · [Методы и source map](../SOURCE_MAP.md) · [Измерения](../reports/VALIDATION.md).

Каждый опыт: **теория → формула → формы/единицы → прогноз → код → измерение → ограниченный вывод**. CPU по умолчанию; без pretrained weights, сети и ComfyUI jobs.


**Связь с MIT:** MIT Lab 3 Part 5 / Q5.1 связывает frozen codec, latent field и sampling. ComfyUI observation и hardware measurements — следующий практический уровень, а не замена MIT вычислений.

[MIT course](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf).


In [ ]:
from pathlib import Path
import sys,math,json,copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
COURSE=next((p/'dit' for p in [Path.cwd(),*Path.cwd().parents] if (p/'dit/labkit/core.py').is_file()),None)
assert COURSE is not None,'Clone the study repository first; see dit/README.md (Colab setup).'
sys.path.insert(0,str(COURSE))
from labkit.core import (native,tiny_model,tiny_inputs,activate_gates,fresh_forward,
  load_k5_block,manual_single,manual_fused,CountingLinearField,tiny_bundle,
  clone_bundle,scalar_bundle,vector_from_bundle,dummy_text,show_source,
  check_sources,save_metrics,figure_path)
torch.set_num_threads(2);torch.manual_seed(42)
LAB='10_comfyui_and_manual_research_protocol';metrics={}
print('Pinned source files checked:',check_sources())


## 1. Где блоки находятся в ComfyUI graph

Используйте существующий [K6 Lite Smoke workflow](../../kandinsky-6/workflows/k6_lite_distill_smoke.json) и [инструкцию Desktop](../../kandinsky-6/COMFYUI_AND_EXPERIMENT_RU.md). Graph визуализирует loader → text encode → joint latent → sampler → video/audio decode → mux. Внутренние SA/CA/gates/policy не становятся отдельными Comfy nodes автоматически.

| UI участок | Разобранный code path | Что можно наблюдать |
|---|---|---|
| Dual text loader / prompt nodes | Text encoder → DiT projections/refiners | Prompt, masks/source type; не внутренние trained gates |
| Empty joint latent / first image | Conditioning / visual_conditioning | Physical shapes, references, frame count |
| K6 Sampler | denoise_loop или π sampler по checkpoint | Calls/steps и pipeline time, при корректном runtime tracing |
| Video/audio VAE decode | Отдельные codecs | RGB/audio duration, reconstruction artifacts |
| Create/Save Video | Postprocess/mux | Container fps/sample rate, sync и trimming |

На данном Mac pretrained inference отложен до 50 GB+ SSD. Этот notebook не скачивает weights, не устанавливает MLX и не отправляет jobs в ComfyUI. Веса полного Pro/SR/Qwen не нужны для CPU labs 00–09.


In [ ]:
workflow=COURSE.parent/'kandinsky-6/workflows/k6_lite_distill_smoke.json'
graph=json.loads(workflow.read_text())
metrics['comfy_static']={'workflow':str(workflow.relative_to(COURSE.parent)),
 'nodes':len(graph['nodes']),'links':len(graph['links']),'jobs_submitted':0,'runtime_verified':False}
for node in graph['nodes']:print(node['id'],node['type'],node.get('title',''))


## 2. Ручной baseline после resource gate

1. Зафиксируйте checkpoint SHA, source revision, architecture class, G/physical channels, text/codec versions, dtype/device. Используйте Lite и уже скачанные совместимые weights; не соглашайтесь автоматически на ненужные Pro/SR downloads.
2. Зафиксируйте seed42, prompt, clip size, FPS, recommended distilled steps/CFG=1. Число steps в UI не доказывает точный transformer NFE: проверьте исполненный sampler и forward hooks/logs.
3. Выполните warmup, затем 3 настоящих runs. Обойдите Comfy cache: неизменённые nodes могут не исполниться. Разделите text encode, transformer/cheap policy, VAE decode, mux и total wall time.
4. Запишите calls и policy counts отдельно. Для native DX schedule считайте rounding по каждому interval; не используйте S×128. В рамках cheap substeps network forward отсутствует, но runtime overhead всё равно измеряется.
5. Для architecture ablations используйте отдельную Python ветку с full weights и hooks. UI смена prompt не изолирует cross gate; CPU lab03 показывает контролируемое отключение A→V/V→A.

На CUDA измерьте synchronized latency, peak allocated и reserved. На MPS point counters before/after не называйте peak; 48 GB unified memory включает macOS/CPU/GPU. Не переносите H100 latency на M3 Max. MLX port потребует совместимости всего computation graph и numerical parity — checkpoint conversion сама её не гарантирует.


## 3. Карточка одного исследования

Гипотеза A: при fixed weights/source inputs отключение A→V уничтожает локальную audio→video dependency, но не обязано улучшить/ухудшить global perceptual quality предсказуемым образом. CPU gradient evidence есть в lab03; реальное влияние на sync требует clips и метрик.

Гипотеза B: при фиксированном trained policy и network NFE увеличение policy substeps уменьшает discretization error до representation floor. На analytic case проверено в lab07; на real model нужны paired outputs и teacher-reference trajectory/quality evidence.

Не меняйте одновременно CFG, checkpoint, resolution и policy density. Не сравнивайте случайную tiny сеть с trained checkpoint как quality baseline. Каждое conclusion должно указывать dataset/input, intervention, budget и measurement units.


In [ ]:
columns=['run_id','source_revision','checkpoint_sha256','model_class','device','dtype','seed',
 'video_shape','audio_shape','grid_points','network_NFE','policy_evaluations_video',
 'policy_evaluations_audio','num_steps_UI','scheduler_shift','final_step_scale',
 'CFG','reference_count','cross_gate_intervention','text_seconds','DiT_seconds',
 'policy_seconds','decode_seconds','mux_seconds','total_seconds',
 'CUDA_peak_allocated_bytes','CUDA_peak_reserved_bytes','MPS_point_counters',
 'metric_name','metric_value','status','limits']
metrics['manual_record_columns']=columns
metrics['research_status']={'CPU_labs':'executed separately by validator','pretrained':'not_run','ComfyUI_runtime':'not_run','MLX_parity':'not_run'}
print(','.join(columns))


## Измерение → вывод

Workflow прочитан статически, составлена схема измерений. Real generation не выполнена. Исследовательский результат сейчас — algebra, native tiny parity, controlled interventions и toy imitation; real AV quality/memory/latency — отдельный следующий этап.


## Самостоятельные опыты

1. Объясните через UI graph, где заканчивается conditioning и начинается neural field evaluation.
2. Напишите ожидаемые NFE/policy counts до запуска, затем сравните с logs.
3. Для одной выбранной гипотезы заполните [общий research template](../../curriculum/EXPERIMENT_TEMPLATE.md).
4. Выберите teacher-reference и no-regression критерии до получения результатов.

Перед изменением запишите прогноз. Сохраняйте baseline, меняйте один фактор, отмечайте seed и tensor layout. Нулевая ошибка на oracle не равна pretrained quality.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Первичные источники

[MIT course / notes](https://diffusion.csail.mit.edu/2026/index.html) · [DiT](https://arxiv.org/abs/2212.09748) · [Flow Matching](https://arxiv.org/abs/2210.02747) · [K5](https://arxiv.org/abs/2511.14993) · [K6 §3.2, §4.5](https://arxiv.org/html/2610.05608v1) · [π-Flow §3.1, §4 / Alg.1](https://arxiv.org/html/2510.14974v1) · [Source hashes / provenance](../sources.json).

Формулы ниже/выше выводятся для явно заданных соглашений. Тела source methods сохраняются; adapters и doubles подписаны. Малые experiments не воспроизводят промышленное обучение.
